# Runtime Agents: Codebase Knowledge Transfer

**Purpose**: This notebook provides a comprehensive walkthrough of the runtime-agents codebase, explaining the architecture, demonstrating key components, and providing an interactive playground for understanding the multi-architecture agent system.

**Target Audience**: Developers who want to understand, extend, or contribute to the runtime-agents system.

**Prerequisites**:
- Python 3.11+
- Basic understanding of async/await patterns
- Familiarity with LLMs and agent concepts
- OpenAI API key (for running examples)

---

## Table of Contents

1. [Introduction](#1-introduction)
2. [Core Components Deep Dive](#2-core-components-deep-dive)
   - 2.1 [LLM Client System](#21-llm-client-system)
   - 2.2 [Tool System](#22-tool-system)
   - 2.3 [Agent Templates and Instances](#23-agent-templates-and-instances)
   - 2.4 [Orchestrator System](#24-orchestrator-system)
   - 2.5 [Session Management](#25-session-management)
   - 2.6 [Agent Factory](#26-agent-factory)
3. [Alternative Architectures](#3-alternative-architectures)
4. [Data Flow Walkthrough](#4-data-flow-walkthrough)
5. [Interactive Playground](#5-interactive-playground)
6. [Performance and Metrics](#6-performance-and-metrics)
7. [Configuration and Extensibility](#7-configuration-and-extensibility)
8. [Common Patterns and Best Practices](#8-common-patterns-and-best-practices)
9. [Troubleshooting Guide](#9-troubleshooting-guide)
10. [Next Steps and Resources](#10-next-steps-and-resources)

---

## 1. Introduction

### What is Runtime Agents?

Runtime Agents is a **multi-architecture agent system** that supports **six different agent orchestration approaches**. Instead of using a single static agent, the system dynamically creates and orchestrates AI agents to handle user requests through conversational chat, file/image uploads, and database connections.

### Key Features

- **6 Agent Architectures**: Template-based, LLM-generated, Compositional, Meta-agent, Hierarchical, and Evolutionary
- **Dynamic Agent Spawning**: Agents are created at runtime based on requirements
- **Tool System**: Extensible tool framework for agent capabilities
- **Session Management**: Persistent sessions with file/image/database support
- **Performance Tracking**: Compare effectiveness across architectures
- **Streamlit UI**: User-friendly chat interface

### Design Principles

1. **Modularity**: Each component is independent and reusable
2. **Extensibility**: Easy to add new architectures, tools, and templates
3. **Protocol-based**: Uses Python protocols for type safety without tight coupling
4. **Async-first**: Built on async/await for efficient I/O operations
5. **Configuration-driven**: Behavior controlled via config files and environment variables

### System Architecture Overview

```mermaid
flowchart TD
    A[User Request] --> B[Streamlit UI]
    B --> C[Session Manager]
    C --> D[Agent Factory]
    D --> E{Select Architecture}
    E -->|template_based| F[Template Orchestrator]
    E -->|llm_generated| G[LLM-Gen Orchestrator]
    E -->|compositional| H[Compositional Orchestrator]
    E -->|meta| I[Meta Orchestrator]
    E -->|hierarchical| J[Hierarchical Orchestrator]
    E -->|evolutionary| K[Evolutionary Orchestrator]
    F --> L[Execute Agents]
    G --> L
    H --> L
    I --> L
    J --> L
    K --> L
    L --> M[Tool Execution]
    M --> N[LLM Calls]
    N --> O[Aggregate Results]
    O --> P[Update Session]
    P --> Q[Display Response]
```

### Six Agent Architectures Comparison

| Architecture | Description | Pros | Cons | Best For |
|-------------|-------------|------|------|----------|
| **Template-Based** | Predefined agent templates with dynamic selection | Predictable, fast, secure, cost-effective | Limited flexibility | Production systems, well-defined use cases |
| **LLM-Generated** | Agents generated dynamically by LLM | Fully adaptive, no predefined roles | Higher cost, less predictable | Research, exploration, novel tasks |
| **Compositional** | Agents built from reusable components | Flexible yet controlled, reusable | Requires component library | Multi-domain applications |
| **Meta-Agent** | Single adaptive agent that adapts per request | Simple, fast, no agent creation overhead | Less specialized | Simple scenarios, latency-critical |
| **Hierarchical** | Tree of agents that decompose tasks | Handles complex workflows, parallelizable | Complex orchestration | Complex multi-step workflows |
| **Evolutionary** | Agents evolve based on performance feedback | Self-improving, learns from experience | Requires feedback, needs time to learn | Long-running systems, personalization |

### Setup: Import Required Modules

Let's start by importing the core modules we'll use throughout this notebook.

In [ ]:
# Standard library imports
import os
import sys
import asyncio
from pathlib import Path
from typing import Dict, List, Any
import json

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

print(f"Project root: {project_root}")
print(f"Python version: {sys.version}")
print(f"OpenAI API key loaded: {'Yes' if os.getenv('OPENAI_API_KEY') else 'No'}")

---

## 2. Core Components Deep Dive

This section explores the fundamental building blocks of the runtime-agents system.

### 2.1 LLM Client System

The LLM client is the foundation for all agent communication. It provides an abstraction over OpenAI's API (and compatible services).

#### Key Components:

1. **`LLMClient` Protocol**: Abstract interface defining the contract
2. **`OpenAIChatClient`**: Concrete implementation using OpenAI's Chat Completions API
3. **`Message` Dataclass**: Represents a chat message with role and content

#### Architecture:

```python
# From runtime_agents/shared/llm.py

class LLMClient(Protocol):
    async def chat(self, messages: List[Message], *, temperature: float, max_tokens: Optional[int]) -> str:
        ...

@dataclass
class Message:
    role: Literal["system", "user", "assistant", "tool"]
    content: str
```

Let's see it in action:

In [ ]:
from runtime_agents.shared.llm import OpenAIChatClient, Message

# Create an LLM client
llm_client = OpenAIChatClient(
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    base_url=os.getenv("OPENAI_BASE_URL", "https://api.openai.com"),
    timeout_s=60.0
)

print(f"LLM Client created:")
print(f"  Model: {llm_client.model}")
print(f"  Base URL: {llm_client.base_url}")
print(f"  Timeout: {llm_client.timeout_s}s")

In [ ]:
# Example: Simple chat interaction
async def test_llm_client():
    messages = [
        Message(role="system", content="You are a helpful assistant that explains technical concepts concisely."),
        Message(role="user", content="What is an agent in AI systems?")
    ]
    
    response = await llm_client.chat(messages, temperature=0.7)
    return response

# Run the async function
response = await test_llm_client()
print("LLM Response:")
print(response)

#### Key Design Decisions:

1. **Protocol-based**: Uses Python's `Protocol` for type safety without inheritance
2. **Async-first**: All LLM calls are async for efficient I/O
3. **HTTP-based**: Uses `httpx` directly instead of OpenAI SDK for simplicity and control
4. **Gateway-compatible**: Works with OpenAI-compatible services (LiteLLM, vLLM, etc.)
5. **Error handling**: Provides helpful error messages for common issues (401, 429, etc.)

#### When to Use:

- Every agent needs an LLM client to communicate with the language model
- The orchestrator passes the same client to all agents for consistency
- You can create multiple clients with different configurations if needed

### 2.2 Tool System

Tools are capabilities that agents can use to interact with the world. The tool system is extensible and protocol-based.

#### Tool Protocol:

```python
@runtime_checkable
class Tool(Protocol):
    name: str
    description: str
    
    async def __call__(self, input: Dict[str, Any]) -> Dict[str, Any]:
        ...
```

#### Built-in Tools:

1. **TimeTool**: Returns current UTC time
2. **HttpGetTool**: Fetches content from URLs
3. **FileReadTool**: Reads uploaded files (text, CSV, JSON, PDF)
4. **FileListTool**: Lists available files in session
5. **DatabaseConnectionTool**: Manages database connections
6. **SchemaIntrospectionTool**: Inspects database schemas
7. **DatabaseQueryTool**: Executes SQL queries safely
8. **ImageAnalysisTool**: Analyzes images with vision models
9. **ImageListTool**: Lists available images in session
10. **RenderPlotTool**: Creates data visualizations
11. **RenderMermaidTool**: Generates mermaid diagrams

Let's explore some tools:

In [ ]:
from runtime_agents.shared.tools import TimeTool, HttpGetTool

# Example 1: TimeTool
time_tool = TimeTool()
print(f"Tool Name: {time_tool.name}")
print(f"Description: {time_tool.description}")

# Execute the tool
result = await time_tool({})
print(f"Result: {result}")

In [ ]:
# Example 2: HttpGetTool
http_tool = HttpGetTool()
print(f"\nTool Name: {http_tool.name}")
print(f"Description: {http_tool.description}")

# Fetch a URL (example.com is safe for testing)
result = await http_tool({"url": "https://example.com"})
print(f"Status: {result['status']}")
print(f"Content preview: {result['text'][:200]}...")

#### Tool Registry

The `ToolRegistry` and `get_default_tools()` function centralize tool creation and management:

In [ ]:
from utils.tool_registry import get_default_tools

# Create default tool set
tools = get_default_tools(
    session_uploads_dir=None,  # No file uploads for this demo
    session_files=None,
    session_images=None,
    db_connection_tool=None
)

print("Available tools:")
for tool_name, tool in tools.items():
    print(f"  - {tool_name}: {tool.description}")

#### Tool Scoping

Each agent template defines which tools it can use. This provides:

1. **Security**: Agents only access tools they need
2. **Clarity**: Clear separation of concerns
3. **Debugging**: Easier to trace which agent used which tool

Example:
```python
# Planner agent only gets planning-related tools
planner_tools = ["time_now", "file_list", "image_list"]

# Analyst agent gets data analysis tools
analyst_tools = ["file_read", "db_schema", "db_query"]
```

### 2.3 Agent Templates and Instances

The template-based architecture uses two key concepts:

1. **`AgentTemplate`**: Immutable specification (like a class definition)
2. **`AgentInstance`**: Runtime entity (like an object instance)

#### AgentTemplate Structure:

```python
@dataclass(frozen=True)
class AgentTemplate:
    key: str              # Unique identifier (e.g., "planner")
    name: str             # Human-readable name
    system_prompt: str    # Role-specific instructions
    tool_names: List[str] # Tools this agent can use
```

#### Four Predefined Templates:

1. **Planner**: Breaks down requests into execution plans
2. **Researcher**: Gathers references and factual details
3. **Analyst**: Analyzes tradeoffs and produces structured reasoning
4. **Writer**: Writes clean, concise outputs

Let's create and explore agent templates:

In [ ]:
from runtime_agents.template_based.agents import AgentTemplate, AgentInstance

# Define the four standard templates
planner_template = AgentTemplate(
    key="planner",
    name="Planner",
    system_prompt="You break down the request into an execution plan and identify missing info.",
    tool_names=["time_now", "file_list", "image_list"]
)

researcher_template = AgentTemplate(
    key="researcher",
    name="Researcher",
    system_prompt="You gather references and factual details. If you need to fetch a URL, use http_get.",
    tool_names=["http_get", "file_read"]
)

analyst_template = AgentTemplate(
    key="analyst",
    name="Analyst",
    system_prompt="You analyze tradeoffs, compare options, and produce structured reasoning.",
    tool_names=["file_read", "db_schema", "db_query"]
)

writer_template = AgentTemplate(
    key="writer",
    name="Writer",
    system_prompt="You write clean, concise outputs tailored to the request.",
    tool_names=["file_read"]
)

# Create a registry
template_registry = {
    "planner": planner_template,
    "researcher": researcher_template,
    "analyst": analyst_template,
    "writer": writer_template
}

print("Agent Template Registry:")
for key, template in template_registry.items():
    print(f"\n{template.name} ({template.key}):")
    print(f"  Prompt: {template.system_prompt[:60]}...")
    print(f"  Tools: {', '.join(template.tool_names)}")

#### Spawning an Agent Instance

An `AgentInstance` is created from a template and given scoped tools:

In [ ]:
# Spawn a planner agent
scoped_tools = {name: tools[name] for name in planner_template.tool_names if name in tools}

planner_agent = AgentInstance(
    template=planner_template,
    llm=llm_client,
    tools=scoped_tools
)

print(f"Agent Instance Created:")
print(f"  Name: {planner_agent.template.name}")
print(f"  Available tools: {list(planner_agent.tools.keys())}")

In [ ]:
# Execute the agent
async def run_planner_agent():
    result = await planner_agent.run(
        user_input="I need to analyze sales data from last quarter and create a summary report.",
        context=None
    )
    return result

result = await run_planner_agent()

print(f"\nAgent Result:")
print(f"  Agent Name: {result.agent_name}")
print(f"  Output: {result.output}")
print(f"  Tool Calls: {len(result.tool_calls)}")

#### Key Features:

1. **Automatic Tool Execution**: Agents detect file references and auto-execute `file_read`
2. **Context Passing**: Agents receive context from previous agents
3. **Session Context**: Agents know about uploaded files, images, and database connections
4. **Immutable Templates**: Templates are frozen dataclasses (version-controlled, pre-approved)
5. **Runtime Instances**: New instance created per request or session

### 2.4 Orchestrator System

The orchestrator coordinates multiple agents to handle complex requests. It follows a **route → spawn → execute → aggregate** pattern.

#### BaseOrchestrator Protocol:

```python
class BaseOrchestrator(Protocol):
    async def run(self, requirement: str) -> Tuple[List[AgentResult], str]:
        """Execute the orchestration and return agent results + final answer."""
        ...
    
    def get_metrics(self) -> ExecutionMetrics:
        """Get performance metrics for the last execution."""
        ...
```

#### Template-Based Orchestrator Workflow:

```mermaid
sequenceDiagram
    participant U as User
    participant O as Orchestrator
    participant R as Router
    participant A1 as Agent 1
    participant A2 as Agent 2
    participant LLM as LLM
    
    U->>O: requirement
    O->>R: route(requirement)
    R->>LLM: Which agents?
    LLM-->>R: [planner, analyst]
    R-->>O: agent keys
    O->>O: spawn(planner)
    O->>A1: run(requirement)
    A1->>LLM: chat
    LLM-->>A1: response
    A1-->>O: AgentResult
    O->>O: spawn(analyst)
    O->>A2: run(requirement, context)
    A2->>LLM: chat
    LLM-->>A2: response
    A2-->>O: AgentResult
    O->>LLM: aggregate results
    LLM-->>O: final answer
    O-->>U: results + final
```

Let's see the orchestrator in action:

In [ ]:
from runtime_agents.template_based.orchestrator import Orchestrator

# Create orchestrator
orchestrator = Orchestrator(
    llm=llm_client,
    registry=template_registry,
    tools=tools,
    session_context=""  # No session context for this demo
)

print("Orchestrator created with:")
print(f"  Agent templates: {list(template_registry.keys())}")
print(f"  Available tools: {list(tools.keys())}")

In [ ]:
# Run a request through the orchestrator
async def run_orchestrator_demo():
    requirement = "Explain the benefits of using async/await in Python for I/O-bound operations."
    
    agent_results, final_answer = await orchestrator.run(requirement)
    
    return agent_results, final_answer

agent_results, final_answer = await run_orchestrator_demo()

print("\n" + "="*80)
print("ORCHESTRATOR EXECUTION RESULTS")
print("="*80)

print(f"\nAgents executed: {len(agent_results)}")
for i, result in enumerate(agent_results, 1):
    print(f"\n--- Agent {i}: {result.agent_name} ---")
    print(f"Output: {result.output[:200]}...")
    print(f"Tool calls: {len(result.tool_calls)}")

print(f"\n--- Final Aggregated Answer ---")
print(final_answer)

#### Routing Logic

The orchestrator uses two routing strategies:

1. **LLM-based routing**: Asks the LLM which agents to use
2. **Heuristic fallback**: Keyword matching if LLM fails

Example heuristics:
```python
if "analyze" in requirement.lower():
    agents.append("analyst")
if "find" or "search" in requirement.lower():
    agents.append("researcher")
```

### 2.5 Session Management

The session system maintains state across multiple requests, including chat history, uploaded files, images, and database connections.

#### Session Structure:

```python
@dataclass
class Session:
    session_id: str
    created_at: str
    last_accessed: str
    chat_history: List[Dict[str, str]]
    files: List[FileMetadata]
    images: List[ImageMetadata]
    db_connections: List[DBConnection]
    agent_state: Dict[str, Any]
    agent_results: List[Dict[str, Any]]
```

#### Key Features:

1. **Local File Storage**: Sessions stored as JSON in `sessions/` directory
2. **File Organization**: Uploads in `uploads/{session_id}/files/` and `uploads/{session_id}/images/`
3. **Portable**: No database required
4. **CRUD Operations**: Create, load, save, delete, list sessions

Let's explore session management:

In [ ]:
from utils.session_manager import SessionManager, Session, FileMetadata

# Create session manager
session_manager = SessionManager(
    sessions_dir=str(project_root / "sessions"),
    uploads_dir=str(project_root / "uploads")
)

# Create a new session
session = session_manager.create_session(session_id="demo_session_001")

print(f"Session created:")
print(f"  ID: {session.session_id}")
print(f"  Created: {session.created_at}")
print(f"  Files: {len(session.files)}")
print(f"  Images: {len(session.images)}")
print(f"  DB Connections: {len(session.db_connections)}")

In [ ]:
# Add messages to chat history
session.add_message("user", "Hello! Can you help me analyze some data?")
session.add_message("assistant", "Of course! I'd be happy to help you analyze your data. Please upload the file or tell me more about what you need.")

# Save session
session_manager.save_session(session)

print(f"\nChat history ({len(session.chat_history)} messages):")
for msg in session.chat_history:
    print(f"  {msg['role']}: {msg['content'][:60]}...")

In [ ]:
# Load session back
loaded_session = session_manager.load_session("demo_session_001")

print(f"\nSession loaded:")
print(f"  ID: {loaded_session.session_id}")
print(f"  Messages: {len(loaded_session.chat_history)}")
print(f"  Last accessed: {loaded_session.last_accessed}")

#### Session Context Building

The system builds rich context for agents:

```python
# Example session context
context = """
Available resources:

Uploaded files available:
  - sales_data.csv (text/csv, 45231 bytes)
  - report_template.md (text/markdown, 1024 bytes)
You can use the 'file_read' tool to read any of these files.

Database connections available:
  - Connection ID: db_abc123, Type: postgresql
    Selected tables: customers, orders, products
You can use 'db_schema' to inspect table structures and 'db_query' to query data.

Recent conversation history:
user: Can you analyze the sales trends?
assistant: I'll analyze the sales data for you...
"""
```

### 2.6 Agent Factory

The `AgentFactory` creates orchestrators based on configuration, enabling runtime switching between architectures.

#### Key Responsibilities:

1. **Load Configuration**: Read `config.yaml` to determine active architecture
2. **Create Orchestrators**: Instantiate the appropriate orchestrator class
3. **Dependency Injection**: Pass LLM client, tools, and session context
4. **Architecture Switching**: Support dynamic architecture changes

Let's explore the factory:

In [ ]:
from utils.agent_factory import AgentFactory
import yaml

# Create factory
factory = AgentFactory(config_path=str(project_root / "config.yaml"))

print("Agent Factory Configuration:")
print(f"  Active agent type: {factory.config.get('agent_type')}")
print(f"\nAvailable architectures:")
print(f"  - template_based")
print(f"  - llm_generated")
print(f"  - compositional")
print(f"  - meta")
print(f"  - hierarchical")
print(f"  - evolutionary")

In [ ]:
# Create a template-based orchestrator using factory
orch = factory.create_orchestrator(
    llm=llm_client,
    tools=tools,
    session_context="",
    agent_type="template_based"  # Override config
)

print(f"\nOrchestrator created:")
print(f"  Type: {type(orch).__name__}")
print(f"  Module: {type(orch).__module__}")

#### Factory Pattern Benefits:

1. **Decoupling**: UI doesn't need to know about specific orchestrator classes
2. **Configuration-driven**: Behavior controlled by config file
3. **Testability**: Easy to create orchestrators with different configurations
4. **Extensibility**: Add new architectures by updating factory

#### How It Works:

```python
# Factory logic (simplified)
if agent_type == "template_based":
    return TemplateOrchestrator(llm, registry, tools, session_context)
elif agent_type == "llm_generated":
    return LLMGeneratedOrchestrator(llm, tools, session_context, max_agents=3)
elif agent_type == "meta":
    return MetaOrchestrator(llm, tools, session_context)
# ... etc
```

---

## 3. Alternative Architectures

Beyond the template-based approach, the system supports five additional architectures.

### 3.1 LLM-Generated Architecture

Instead of predefined templates, this architecture uses an LLM to generate agent specifications dynamically.

#### How It Works:

1. **Analyze Request**: LLM analyzes the user requirement
2. **Generate Spec**: LLM creates agent specification (role, prompt, tools)
3. **Create Agent**: System instantiates agent from generated spec
4. **Execute**: Agent runs with dynamically assigned capabilities

#### Agent Spec Structure:

```python
@dataclass
class AgentSpec:
    name: str              # Generated role name
    system_prompt: str     # Generated instructions
    tool_names: List[str]  # Selected tools
```

#### Example Generation Prompt:

```
Given this user requirement: "Analyze sales trends and create a report"

Generate 2-3 agent specifications. For each agent, provide:
1. Role name (e.g., "Data Analyst", "Report Writer")
2. System prompt describing their responsibilities
3. List of tools they need from: [file_read, db_query, http_get, ...]

Return as JSON array.
```

Let's see it in action:

In [ ]:
from runtime_agents_llm_generated.orchestrator import LLMGeneratedOrchestrator

# Create LLM-generated orchestrator
llm_gen_orch = LLMGeneratedOrchestrator(
    llm=llm_client,
    available_tools=tools,
    session_context="",
    max_agents=2,
    temperature=0.7
)

print("LLM-Generated Orchestrator created")
print(f"  Max agents: {llm_gen_orch.max_agents}")
print(f"  Temperature: {llm_gen_orch.temperature}")

In [ ]:
# Run a request (this will generate agents dynamically)
async def run_llm_generated_demo():
    requirement = "Research the history of Python programming language and summarize key milestones."
    
    agent_results, final_answer = await llm_gen_orch.run(requirement)
    
    return agent_results, final_answer

agent_results, final_answer = await run_llm_generated_demo()

print("\n" + "="*80)
print("LLM-GENERATED ARCHITECTURE RESULTS")
print("="*80)

print(f"\nDynamically generated agents: {len(agent_results)}")
for i, result in enumerate(agent_results, 1):
    print(f"\n--- Agent {i}: {result.agent_name} ---")
    print(f"Output: {result.output[:150]}...")

print(f"\n--- Final Answer ---")
print(final_answer[:300] + "...")

#### Pros and Cons:

**Pros:**
- Fully adaptive to any task
- No need to predefine agent roles
- Can handle novel, unexpected requests

**Cons:**
- Higher cost (extra LLM call to generate specs)
- Less predictable behavior
- Slower (generation overhead)
- Harder to version control and audit

**Best For:**
- Research and exploration
- Prototyping new capabilities
- Tasks with unpredictable requirements

### 3.2 Meta-Agent Architecture

Uses a **single adaptive agent** that dynamically adjusts its prompt and tool selection per request.

#### Key Components:

1. **DynamicPromptBuilder**: Generates system prompt based on request
2. **DynamicToolSelector**: Chooses relevant tools for the task
3. **MetaAgent**: Single agent with dynamic configuration

#### Architecture:

```mermaid
flowchart LR
    A[Request] --> B[Prompt Builder]
    A --> C[Tool Selector]
    B --> D[Meta Agent]
    C --> D
    D --> E[LLM]
    E --> F[Response]
```

Let's explore:

In [ ]:
from runtime_agents_meta.orchestrator import MetaOrchestrator

# Create meta-agent orchestrator
meta_orch = MetaOrchestrator(
    llm=llm_client,
    available_tools=tools,
    session_context=""
)

print("Meta-Agent Orchestrator created")
print("  Uses single adaptive agent")
print("  Dynamic prompt building")
print("  Dynamic tool selection")

In [ ]:
# Run a request
async def run_meta_agent_demo():
    requirement = "What are the key differences between async and sync programming?"
    
    agent_results, final_answer = await meta_orch.run(requirement)
    
    return agent_results, final_answer

agent_results, final_answer = await run_meta_agent_demo()

print("\n" + "="*80)
print("META-AGENT ARCHITECTURE RESULTS")
print("="*80)

print(f"\nAgents used: {len(agent_results)} (single meta-agent)")
print(f"\nFinal Answer:")
print(final_answer)

#### Pros and Cons:

**Pros:**
- Simple architecture (one agent)
- Fast (no agent spawning overhead)
- Lower cost (fewer LLM calls)
- Easy to understand and debug

**Cons:**
- Less specialized than multi-agent
- Can't parallelize work
- Single point of failure

**Best For:**
- Simple, straightforward tasks
- Latency-critical applications
- Cost-sensitive scenarios

### 3.3 Other Architectures Overview

#### Compositional Architecture

Builds agents by composing reusable components from a library.

**Components:**
- Skills (e.g., data analysis, web research)
- Behaviors (e.g., thorough, quick, creative)
- Tool sets (e.g., database tools, file tools)

**Example:**
```python
# Compose an agent from components
agent = compose(
    skills=["data_analysis", "visualization"],
    behavior="thorough",
    tools=["file_read", "db_query", "render_plot"]
)
```

**Best For:** Multi-domain applications with reusable capabilities

---

#### Hierarchical Architecture

Organizes agents in a tree structure with parent-child relationships.

**Structure:**
```
Parent Agent (Coordinator)
├── Child Agent 1 (Research)
│   ├── Grandchild 1.1 (Web Search)
│   └── Grandchild 1.2 (File Read)
├── Child Agent 2 (Analysis)
└── Child Agent 3 (Writing)
```

**Features:**
- Task decomposition
- Parallel execution
- Hierarchical aggregation

**Best For:** Complex multi-step workflows

---

#### Evolutionary Architecture

Maintains a pool of agent configurations that evolve based on performance.

**Process:**
1. **Selection**: Choose best-performing agent from pool
2. **Execution**: Run selected agent
3. **Evaluation**: Calculate fitness score
4. **Mutation**: Create variations of successful agents
5. **Crossover**: Combine successful agent traits
6. **Update Pool**: Add new configurations

**Best For:** Long-running systems, personalized agents

### Architecture Comparison Matrix

| Feature | Template | LLM-Gen | Compositional | Meta | Hierarchical | Evolutionary |
|---------|----------|---------|---------------|------|--------------|-------------|
| **Predictability** | ⭐⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ |
| **Flexibility** | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Speed** | ⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐ |
| **Cost** | ⭐⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐ |
| **Complexity** | ⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| **Specialization** | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| **Learning** | ❌ | ❌ | ❌ | ❌ | ❌ | ✅ |
| **Parallel Execution** | ❌ | ❌ | ✅ | ❌ | ✅ | ✅ |

**Legend:** ⭐ = Rating (more stars = better), ✅ = Yes, ❌ = No

---

## 4. Data Flow Walkthrough

Let's trace a complete request through the system to understand how data flows.

### End-to-End Flow Diagram

```mermaid
sequenceDiagram
    participant User
    participant UI as Streamlit UI
    participant SM as Session Manager
    participant AF as Agent Factory
    participant Orch as Orchestrator
    participant Router
    participant Agent1
    participant Agent2
    participant Tool
    participant LLM
    
    User->>UI: Enter message
    UI->>SM: Load/create session
    SM-->>UI: Session data
    UI->>UI: Build session context
    UI->>AF: Create orchestrator
    AF-->>UI: Orchestrator instance
    UI->>Orch: run(requirement)
    Orch->>Router: route(requirement)
    Router->>LLM: Which agents?
    LLM-->>Router: [analyst, writer]
    Router-->>Orch: agent keys
    
    Orch->>Orch: spawn(analyst)
    Orch->>Agent1: run(requirement, context)
    Agent1->>LLM: chat(messages)
    LLM-->>Agent1: response
    Agent1->>Tool: file_read(filename)
    Tool-->>Agent1: file content
    Agent1->>LLM: chat with file content
    LLM-->>Agent1: analysis
    Agent1-->>Orch: AgentResult
    
    Orch->>Orch: accumulate context
    Orch->>Orch: spawn(writer)
    Orch->>Agent2: run(requirement, context)
    Agent2->>LLM: chat(messages)
    LLM-->>Agent2: response
    Agent2-->>Orch: AgentResult
    
    Orch->>LLM: aggregate(results)
    LLM-->>Orch: final answer
    Orch-->>UI: results + final
    UI->>SM: Save session
    SM-->>UI: Saved
    UI-->>User: Display response
```

### Step-by-Step Execution Trace

Let's create a detailed trace of a request with logging:

In [ ]:
import logging
from runtime_agents.shared.logger import get_logger

# Enable debug logging
logger = get_logger(__name__)
logger.setLevel(logging.DEBUG)

# Create a simple handler to see logs in notebook
handler = logging.StreamHandler()
handler.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(levelname)s] %(name)s: %(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)

print("Logging enabled at DEBUG level")

In [ ]:
# Trace a complete request
async def trace_complete_request():
    print("\n" + "="*80)
    print("STEP-BY-STEP EXECUTION TRACE")
    print("="*80 + "\n")
    
    # Step 1: Create session
    print("[STEP 1] Creating session...")
    session = session_manager.create_session(session_id="trace_demo")
    print(f"  ✓ Session created: {session.session_id}\n")
    
    # Step 2: Add user message
    print("[STEP 2] Adding user message...")
    user_msg = "Explain the benefits of protocol-based design in Python."
    session.add_message("user", user_msg)
    print(f"  ✓ Message added: {user_msg}\n")
    
    # Step 3: Build session context
    print("[STEP 3] Building session context...")
    context = "Available resources:\n  (No files or databases in this demo)"
    print(f"  ✓ Context built: {len(context)} chars\n")
    
    # Step 4: Create orchestrator via factory
    print("[STEP 4] Creating orchestrator via factory...")
    orch = factory.create_orchestrator(
        llm=llm_client,
        tools=tools,
        session_context=context,
        agent_type="template_based"
    )
    print(f"  ✓ Orchestrator created: {type(orch).__name__}\n")
    
    # Step 5: Run orchestrator
    print("[STEP 5] Running orchestrator...")
    print("  (Watch for routing, spawning, execution logs below)\n")
    
    agent_results, final_answer = await orch.run(user_msg)
    
    print(f"\n  ✓ Orchestrator completed\n")
    
    # Step 6: Process results
    print("[STEP 6] Processing results...")
    print(f"  Agents executed: {len(agent_results)}")
    for i, result in enumerate(agent_results, 1):
        print(f"    {i}. {result.agent_name}: {len(result.output)} chars, {len(result.tool_calls)} tool calls")
    print(f"  Final answer: {len(final_answer)} chars\n")
    
    # Step 7: Save to session
    print("[STEP 7] Saving to session...")
    session.add_message("assistant", final_answer)
    session_manager.save_session(session)
    print(f"  ✓ Session saved\n")
    
    print("="*80)
    print("TRACE COMPLETE")
    print("="*80)
    
    return agent_results, final_answer

# Run the trace
agent_results, final_answer = await trace_complete_request()

### Context Accumulation

One of the key features is how context accumulates as agents execute:

In [ ]:
# Demonstrate context accumulation
print("Context Accumulation Example:\n")
print("Initial context (from session):")
print("  - Uploaded files: file1.csv, file2.json")
print("  - Database: PostgreSQL connection")
print("  - Recent chat: 3 messages\n")

print("After Agent 1 (Analyst) executes:")
print("  Context += '[Analyst]\\n' + analyst_output")
print("  Now contains: session context + analyst findings\n")

print("After Agent 2 (Writer) executes:")
print("  Context += '[Writer]\\n' + writer_output")
print("  Now contains: session + analyst + writer\n")

print("Aggregation step:")
print("  LLM receives ALL accumulated context")
print("  Produces final synthesized answer")

---

## 5. Interactive Playground

Now it's your turn! This section provides hands-on exercises to deepen your understanding.

### 5.1 Create a Simple Agent

**Exercise**: Define a custom agent template for a "Code Reviewer" role.

In [ ]:
# TODO: Create a CodeReviewer agent template
# Hints:
# - key: "code_reviewer"
# - name: "Code Reviewer"
# - system_prompt: Should focus on code quality, best practices, potential bugs
# - tool_names: ["file_read"] to read code files

code_reviewer_template = AgentTemplate(
    key="code_reviewer",
    name="Code Reviewer",
    system_prompt="You are an expert code reviewer. Analyze code for quality, best practices, potential bugs, and suggest improvements. Be constructive and specific.",
    tool_names=["file_read"]
)

print("Custom Agent Template Created:")
print(f"  Name: {code_reviewer_template.name}")
print(f"  Key: {code_reviewer_template.key}")
print(f"  Tools: {code_reviewer_template.tool_names}")
print(f"  Prompt: {code_reviewer_template.system_prompt[:80]}...")

In [ ]:
# Spawn and test the code reviewer agent
async def test_code_reviewer():
    # Create scoped tools
    scoped_tools = {name: tools[name] for name in code_reviewer_template.tool_names if name in tools}
    
    # Spawn agent
    reviewer_agent = AgentInstance(
        template=code_reviewer_template,
        llm=llm_client,
        tools=scoped_tools
    )
    
    # Test with a code review request
    code_sample = '''
def calculate_total(items):
    total = 0
    for item in items:
        total = total + item['price'] * item['quantity']
    return total
'''
    
    request = f"Review this Python function:\n{code_sample}"
    
    result = await reviewer_agent.run(request, context=None)
    
    return result

review_result = await test_code_reviewer()

print("\nCode Review Result:")
print("="*80)
print(review_result.output)
print("="*80)

### 5.2 Build a Custom Tool

**Exercise**: Implement a custom tool that calculates statistics from a list of numbers.

In [ ]:
from dataclasses import dataclass
from typing import Dict, Any
import statistics

# TODO: Implement a StatisticsTool
# Input: {"numbers": [1, 2, 3, 4, 5]}
# Output: {"mean": 3.0, "median": 3, "stdev": 1.58, "min": 1, "max": 5}

@dataclass
class StatisticsTool:
    name: str = "calculate_statistics"
    description: str = "Calculate statistics (mean, median, stdev, min, max) from a list of numbers. Input: {numbers: List[float]}."
    
    async def __call__(self, input: Dict[str, Any]) -> Dict[str, Any]:
        numbers = input.get("numbers", [])
        
        if not numbers:
            return {"error": "No numbers provided"}
        
        if not isinstance(numbers, list):
            return {"error": "Input must be a list of numbers"}
        
        try:
            result = {
                "count": len(numbers),
                "mean": statistics.mean(numbers),
                "median": statistics.median(numbers),
                "min": min(numbers),
                "max": max(numbers)
            }
            
            # Only calculate stdev if we have 2+ numbers
            if len(numbers) >= 2:
                result["stdev"] = statistics.stdev(numbers)
            
            return result
        except Exception as e:
            return {"error": f"Calculation failed: {str(e)}"}

# Test the tool
stats_tool = StatisticsTool()
print(f"Custom Tool Created: {stats_tool.name}")
print(f"Description: {stats_tool.description}\n")

# Test it
test_data = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
result = await stats_tool({"numbers": test_data})

print(f"Test input: {test_data}")
print(f"Result: {json.dumps(result, indent=2)}")

In [ ]:
# Register the tool and use it with an agent
custom_tools = tools.copy()
custom_tools["calculate_statistics"] = stats_tool

# Create a "Data Scientist" agent that can use this tool
data_scientist_template = AgentTemplate(
    key="data_scientist",
    name="Data Scientist",
    system_prompt="You are a data scientist. Analyze numerical data and provide insights.",
    tool_names=["calculate_statistics", "time_now"]
)

# Spawn agent
scoped_tools = {name: custom_tools[name] for name in data_scientist_template.tool_names if name in custom_tools}
data_scientist = AgentInstance(
    template=data_scientist_template,
    llm=llm_client,
    tools=scoped_tools
)

print("Data Scientist agent created with custom statistics tool")
print(f"Available tools: {list(scoped_tools.keys())}")

### 5.3 Test Different Orchestrators

**Exercise**: Run the same query through different architectures and compare results.

We'll use the sample data from the repository:

In [ ]:
import pandas as pd

# Load sample data
sample_data_path = project_root / "sample_data" / "school_electricity_access.csv"

if sample_data_path.exists():
    df = pd.read_csv(sample_data_path)
    print("Sample Data Loaded:")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    print(f"\nFirst few rows:")
    print(df.head())
else:
    print("Sample data not found. Using synthetic data.")
    df = pd.DataFrame({
        'country': ['USA', 'UK', 'Germany', 'France', 'Japan'],
        'electricity_access': [100, 100, 100, 100, 100],
        'year': [2020, 2020, 2020, 2020, 2020]
    })

In [ ]:
# Compare architectures
async def compare_architectures():
    query = "What insights can you provide about the electricity access data?"
    
    results = {}
    
    # Test Template-Based
    print("\n" + "="*80)
    print("Testing TEMPLATE-BASED Architecture")
    print("="*80)
    
    template_orch = factory.create_orchestrator(
        llm=llm_client,
        tools=tools,
        session_context="",
        agent_type="template_based"
    )
    
    agent_results, final = await template_orch.run(query)
    results['template_based'] = {
        'agents_used': len(agent_results),
        'agent_names': [r.agent_name for r in agent_results],
        'final_answer': final[:200] + "..."
    }
    print(f"✓ Completed: {len(agent_results)} agents used")
    
    # Test Meta-Agent
    print("\n" + "="*80)
    print("Testing META-AGENT Architecture")
    print("="*80)
    
    meta_orch = factory.create_orchestrator(
        llm=llm_client,
        tools=tools,
        session_context="",
        agent_type="meta"
    )
    
    agent_results, final = await meta_orch.run(query)
    results['meta'] = {
        'agents_used': len(agent_results),
        'agent_names': [r.agent_name for r in agent_results],
        'final_answer': final[:200] + "..."
    }
    print(f"✓ Completed: {len(agent_results)} agents used")
    
    return results

comparison_results = await compare_architectures()

# Display comparison
print("\n" + "="*80)
print("ARCHITECTURE COMPARISON")
print("="*80)

for arch_name, result in comparison_results.items():
    print(f"\n{arch_name.upper()}:")
    print(f"  Agents used: {result['agents_used']}")
    print(f"  Agent names: {', '.join(result['agent_names'])}")
    print(f"  Answer preview: {result['final_answer']}")

---

## 6. Performance and Metrics

The system tracks performance metrics to compare architecture effectiveness.

### ExecutionMetrics Structure

```python
@dataclass
class ExecutionMetrics:
    agent_type: str
    execution_time: float          # Seconds
    token_usage: Dict[str, int]    # {input_tokens, output_tokens}
    cost_estimate: float           # USD
    num_agents_spawned: int
    tool_calls_count: int
```

### Performance Tracker

The `PerformanceTracker` collects and stores metrics:

In [ ]:
from utils.performance_tracker import PerformanceTracker

# Create tracker
perf_tracker = PerformanceTracker(metrics_file=str(project_root / "performance_metrics.json"))

# Record some sample metrics
perf_tracker.record_execution(
    agent_type="template_based",
    execution_time=2.5,
    token_usage={"input_tokens": 500, "output_tokens": 300},
    cost_estimate=0.002,
    num_agents_spawned=2,
    tool_calls_count=1,
    success=True
)

perf_tracker.record_execution(
    agent_type="meta",
    execution_time=1.8,
    token_usage={"input_tokens": 400, "output_tokens": 250},
    cost_estimate=0.0015,
    num_agents_spawned=1,
    tool_calls_count=0,
    success=True
)

perf_tracker.save_metrics()

print("Performance metrics recorded")

In [ ]:
# Get comparison statistics
comparison_stats = perf_tracker.get_comparison_stats()

print("\nPerformance Comparison:")
print("="*80)

if comparison_stats:
    for agent_type, stats in comparison_stats.items():
        print(f"\n{agent_type.upper()}:")
        for metric, value in stats.items():
            print(f"  {metric}: {value}")
else:
    print("No metrics available yet. Run some queries to collect data.")

### Metrics Visualization

You can visualize metrics to compare architectures:

In [ ]:
# Create a simple comparison table
if comparison_stats:
    import pandas as pd
    
    df_metrics = pd.DataFrame(comparison_stats).T
    print("\nMetrics Comparison Table:")
    print(df_metrics)
else:
    print("Run more queries to collect metrics for visualization")

---

## 7. Configuration and Extensibility

The system is designed to be easily extended with new architectures, tools, and templates.

### config.yaml Structure

```yaml
agent_type: template_based  # Active architecture

# Architecture-specific settings
template_based:
  use_llm_routing: true
  fallback_to_heuristics: true

llm_generated:
  temperature: 0.7
  max_agents: 3
  generation_model: gpt-4o-mini

compositional:
  component_library_path: components/
  enable_behavior_styles: true

meta:
  adaptive_prompting: true
  dynamic_tool_selection: true

hierarchical:
  max_depth: 3
  parallel_execution: true

evolutionary:
  pool_size: 10
  mutation_rate: 0.1
  fitness_threshold: 0.7
```

In [ ]:
# Read and display current config
config_path = project_root / "config.yaml"

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Current Configuration:")
print(json.dumps(config, indent=2))

### How to Add a New Agent Template

**Step 1**: Define the template

```python
security_auditor = AgentTemplate(
    key="security_auditor",
    name="Security Auditor",
    system_prompt="You audit code and systems for security vulnerabilities.",
    tool_names=["file_read", "http_get"]
)
```

**Step 2**: Add to registry in `agent_factory.py`

```python
registry = {
    "planner": planner_template,
    "researcher": researcher_template,
    "analyst": analyst_template,
    "writer": writer_template,
    "security_auditor": security_auditor,  # New!
}
```

**Step 3**: Update routing logic (optional)

```python
# In orchestrator.py routing heuristics
if any(w in req for w in ["security", "vulnerability", "audit"]):
    keys.append("security_auditor")
```

### How to Implement a New Orchestrator

**Step 1**: Create new module (e.g., `runtime_agents_custom/`)

**Step 2**: Implement `BaseOrchestrator` protocol

```python
from runtime_agents.shared.base import BaseOrchestrator, ExecutionMetrics

@dataclass
class CustomOrchestrator(BaseOrchestrator):
    llm: LLMClient
    available_tools: Dict[str, Tool]
    session_context: str = ""
    
    async def run(self, requirement: str) -> Tuple[List[AgentResult], str]:
        # Your custom orchestration logic
        pass
    
    def get_metrics(self) -> ExecutionMetrics:
        # Return performance metrics
        pass
```

**Step 3**: Add to `AgentFactory`

```python
# In agent_factory.py
elif agent_type == "custom":
    return CustomOrchestrator(
        llm=llm,
        available_tools=tools,
        session_context=session_context
    )
```

**Step 4**: Add to config.yaml

```yaml
custom:
  setting1: value1
  setting2: value2
```

---

## 8. Common Patterns and Best Practices

### Tool Scoping Strategies

**Principle**: Give agents only the tools they need.

```python
# ✅ Good: Scoped tools
planner_tools = ["time_now", "file_list"]  # Planning-related only
analyst_tools = ["file_read", "db_query"]  # Data access only

# ❌ Bad: All tools to all agents
all_agents_tools = list(tools.keys())  # Security risk, unclear intent
```

**Benefits**:
1. Security: Agents can't access inappropriate capabilities
2. Clarity: Clear separation of concerns
3. Debugging: Easier to trace which agent used which tool

### Context Passing Patterns

**Pattern 1: Session Context (Static)**

```python
# Built once at request start
session_context = """
Available resources:
- Files: data.csv, report.md
- Database: PostgreSQL (customers, orders)
- Recent chat: 3 messages
"""
```

**Pattern 2: Agent Context (Accumulating)**

```python
# Grows as agents execute
context = session_context
for agent in agents:
    result = agent.run(requirement, context=context)
    context += f"\n\n[{agent.name}]\n{result.output}"
```

**Pattern 3: Selective Context**

```python
# Only pass relevant context
if agent.needs_file_context:
    context = session_context + file_info
else:
    context = session_context
```

### Error Handling Approaches

**Pattern 1: Tool-level Error Handling**

```python
async def __call__(self, input: Dict[str, Any]) -> Dict[str, Any]:
    try:
        # Tool logic
        return {"result": data}
    except Exception as e:
        return {"error": str(e)}  # Return error, don't raise
```

**Pattern 2: Agent-level Error Handling**

```python
try:
    result = await agent.run(requirement)
except Exception as e:
    logger.error(f"Agent {agent.name} failed: {e}")
    result = AgentResult(
        agent_name=agent.name,
        output=f"Error: {str(e)}",
        tool_calls=[]
    )
```

**Pattern 3: Orchestrator-level Error Handling**

```python
try:
    agent_results, final = await orchestrator.run(requirement)
except Exception as e:
    # Log error, record metrics, return graceful failure
    return [], f"System error: {str(e)}"
```

### Logging and Debugging Tips

**Tip 1: Use structured logging**

```python
logger.debug(f"[AGENT:{agent.name}] Starting execution")
logger.info(f"[ROUTER] Selected agents: {agent_keys}")
logger.warning(f"[TOOL:{tool.name}] Slow execution: {duration}s")
```

**Tip 2: Log at decision points**

```python
# Before routing
logger.debug(f"[ROUTER] Requirement: {requirement[:100]}")

# After routing
logger.info(f"[ROUTER] Selected: {keys}")

# Before tool execution
logger.debug(f"[AGENT] Executing tool: {tool_name}")
```

**Tip 3: Use log levels appropriately**

- `DEBUG`: Detailed execution flow
- `INFO`: High-level decisions and completions
- `WARNING`: Unexpected but handled situations
- `ERROR`: Failures requiring attention

---

## 9. Troubleshooting Guide

### Common Issues and Solutions

#### Issue 1: "Missing OpenAI API key"

**Symptoms**: `RuntimeError: Missing OpenAI API key.`

**Solutions**:
1. Check `.env` file exists and has `OPENAI_API_KEY=your-key`
2. Verify `load_dotenv()` is called before creating LLM client
3. Check environment variable: `os.getenv("OPENAI_API_KEY")`

```python
# Debug
print(f"API key loaded: {'Yes' if os.getenv('OPENAI_API_KEY') else 'No'}")
```

---

#### Issue 2: "Agent not found in registry"

**Symptoms**: Router returns invalid agent keys

**Solutions**:
1. Check template registry has the agent key
2. Verify routing logic (LLM or heuristic) returns valid keys
3. Add fallback logic

```python
# Debug
print(f"Registry keys: {list(registry.keys())}")
print(f"Router returned: {agent_keys}")
print(f"Valid keys: {[k for k in agent_keys if k in registry]}")
```

---

#### Issue 3: "Tool execution fails"

**Symptoms**: Tool returns error dict

**Solutions**:
1. Check tool input format matches expected schema
2. Verify tool dependencies (files exist, DB connected, etc.)
3. Check tool error messages for details

```python
# Debug
result = await tool({"filename": "test.csv"})
if "error" in result:
    print(f"Tool error: {result['error']}")
```

---

#### Issue 4: "Session not persisting"

**Symptoms**: Chat history lost between requests

**Solutions**:
1. Verify `session_manager.save_session()` is called
2. Check `sessions/` directory exists and is writable
3. Verify session ID is consistent

```python
# Debug
print(f"Session ID: {session.session_id}")
print(f"Sessions dir: {session_manager.sessions_dir}")
print(f"Sessions: {session_manager.list_sessions()}")
```

---

#### Issue 5: "Slow execution"

**Symptoms**: Requests take too long

**Solutions**:
1. Check number of agents spawned (reduce if too many)
2. Use faster architecture (meta-agent vs template-based)
3. Reduce LLM temperature for faster responses
4. Enable parallel execution (hierarchical architecture)

```python
# Debug
import time
start = time.time()
result = await orchestrator.run(requirement)
print(f"Execution time: {time.time() - start:.2f}s")
```

### Debug Log Interpretation

**Example log sequence:**

```
[INFO] [ORCHESTRATOR] Starting execution for requirement: Analyze sales...
[DEBUG] [ROUTER] Starting routing for requirement: Analyze sales...
[DEBUG] [ROUTER] Available agents: ['planner', 'researcher', 'analyst', 'writer']
[DEBUG] [ROUTER] Attempting LLM-based routing...
[DEBUG] [ROUTER] LLM routing response: analyst,writer
[INFO] [ROUTER] Selected agents via LLM: ['analyst', 'writer']
[INFO] [ORCHESTRATOR] Execution plan: ['analyst', 'writer']
[INFO] [ORCHESTRATOR] Executing agent 1/2: analyst
[DEBUG] [SPAWN] Spawning agent 'analyst' (Analyst) with tools: ['file_read', 'db_query']
[DEBUG] [AGENT:Analyst] Starting execution
[DEBUG] [AGENT:Analyst] Calling LLM with 2 messages
[INFO] [AGENT:Analyst] Auto-detected file from user request: sales_data.csv
[DEBUG] [AGENT:Analyst] Reading file: sales_data.csv
[INFO] [ORCHESTRATOR] Agent 'analyst' completed. Output length: 523 chars
[INFO] [ORCHESTRATOR] Executing agent 2/2: writer
...
[INFO] [ORCHESTRATOR] All agents completed, aggregating results...
[INFO] [ORCHESTRATOR] Aggregation complete. Final answer length: 892 chars
```

**What to look for:**
- Routing decisions: Which agents were selected and why
- Agent execution: Which agents ran and in what order
- Tool calls: Which tools were executed and with what input
- Timing: Where time is being spent
- Errors: Any warnings or errors in the flow

### Testing Strategies

**Strategy 1: Unit test individual components**

```python
# Test LLM client
async def test_llm_client():
    messages = [Message("user", "Hello")]
    response = await llm_client.chat(messages)
    assert len(response) > 0

# Test tool
async def test_time_tool():
    tool = TimeTool()
    result = await tool({})
    assert "utc_now" in result
```

**Strategy 2: Integration test agent execution**

```python
async def test_agent_execution():
    agent = AgentInstance(template, llm, tools)
    result = await agent.run("Test request")
    assert result.agent_name == template.name
    assert len(result.output) > 0
```

**Strategy 3: End-to-end test orchestrator**

```python
async def test_orchestrator():
    orch = Orchestrator(llm, registry, tools)
    agent_results, final = await orch.run("Test requirement")
    assert len(agent_results) > 0
    assert len(final) > 0
```

---

## 10. Next Steps and Resources

### Documentation Files

The repository includes comprehensive documentation:

1. **`docs/understanding.md`**: Detailed system architecture and workflow
2. **`docs/improvement_ideas.md`**: POC roadmap and future enhancements
3. **`docs/alternative_approaches.md`**: Detailed comparison of all architectures
4. **`docs/implementation_summary.md`**: Implementation details and decisions
5. **`docs/quick_reference.md`**: Quick reference guide
6. **`README.md`**: Getting started and overview

### Suggested Exercises

**Beginner:**
1. Create a custom agent template for a new role (e.g., "Translator", "Summarizer")
2. Implement a simple tool (e.g., calculator, string formatter)
3. Modify routing heuristics to add new keywords

**Intermediate:**
1. Add a new orchestrator architecture
2. Implement parallel agent execution
3. Add caching for LLM responses
4. Create a custom session storage backend (e.g., SQLite)

**Advanced:**
1. Implement the evolutionary architecture fully
2. Add support for streaming responses
3. Implement agent memory across sessions
4. Add support for multi-modal inputs (images + text)
5. Create a distributed orchestrator (agents on different machines)

### Areas for Contribution

**High Priority:**
- Complete implementation of alternative architectures
- Add comprehensive test suite
- Improve tool auto-execution logic
- Add OpenAI function calling support

**Medium Priority:**
- Add more built-in tools
- Implement parallel agent execution
- Add response caching
- Improve error handling and recovery

**Nice to Have:**
- Add visualization tools for agent execution
- Create agent playground UI
- Add support for other LLM providers
- Implement agent versioning

### Further Reading

**Agent Systems:**
- [LangChain Documentation](https://python.langchain.com/docs/)
- [AutoGPT Architecture](https://github.com/Significant-Gravitas/AutoGPT)
- [Microsoft AutoGen](https://microsoft.github.io/autogen/)

**Python Async:**
- [Python asyncio Documentation](https://docs.python.org/3/library/asyncio.html)
- [Real Python: Async IO](https://realpython.com/async-io-python/)

**Design Patterns:**
- [Protocol-based Programming in Python](https://peps.python.org/pep-0544/)
- [Factory Pattern](https://refactoring.guru/design-patterns/factory-method)

---

## Conclusion

You've completed the runtime-agents knowledge transfer notebook! You should now understand:

✅ The six agent architectures and when to use each
✅ Core components: LLM client, tools, agents, orchestrators
✅ How data flows through the system
✅ Session management and state persistence
✅ How to extend the system with new agents, tools, and architectures
✅ Common patterns and best practices
✅ Troubleshooting techniques

**Next Steps:**
1. Try the exercises in Section 5
2. Read the documentation files
3. Experiment with different architectures
4. Contribute improvements to the codebase

**Questions or Issues?**
- Check the troubleshooting guide (Section 9)
- Review the documentation files
- Examine the source code with your new understanding

Happy coding! 🚀